In [ ]:
import pandas as pd
import numpy as np

# On crée 10 lignes de données réalistes
data_test = {
    'numero_dpe': ['DPE001', 'DPE001', 'DPE003', 'DPE004', 'DPE005', 'DPE006', 'DPE007', 'DPE008', 'DPE009', 'DPE010'],
    'numero_rpls_logement': [np.nan, 'RPLS-99', np.nan, np.nan, 'RPLS-88', np.nan, np.nan, np.nan, 'RPLS-77', np.nan],
    'numero_immatriculation_copropriete': ['COPRO-1', np.nan, 'COPRO-2', np.nan, np.nan, 'COPRO-3', np.nan, 'COPRO-4', np.nan, np.nan],
    'adresse_brut': ['10 rue de Rivoli', '5 av Foch', '12 rue de la Paix', '3 bld Haussmann', '20 rue de Bercy', '8 rue du Bac', '1 av de Lyon', '90 rue de Paris', '45 rue du Commerce', '7 rue de Rennes'],
    'code_postal_brut': [75001, 75016, 75001, 75008, 75012, 75007, 69000, 75012, 75015, 75006],
    'date_etablissement_dpe': ['2023-01-10', '2022-05-15', '2023-08-20', '2021-12-01', '2023-03-12', '2022-11-30', '2023-01-05', '2023-06-25', '2022-09-10', '2023-02-28'],
    'date_fin_validite_dpe': ['2033-01-10', '2032-05-15', '2033-08-20', '2031-12-01', '2033-03-12', '2032-11-30', '2033-01-05', '2033-06-25', '2032-09-10', '2033-02-28'],
    'annee_construction': [1950, 1970, 1900, 1985, 2010, 1960, 1975, 1955, 2015, 1930],
    'etiquette_dpe': ['G', 'C', 'A', 'E', 'B', 'D', 'F', 'G', 'A', 'E'],
    'etiquette_ges': ['F', 'B', 'A', 'D', 'B', 'C', 'E', 'F', 'A', 'D'],
    'surface_habitable_logement': [45.5, 120.0, 32.0, 85.0, 65.0, 50.0, 75.0, 42.0, 95.0, 28.0],
    'conso_5_usages_par_m2_ef': [450.2, 120.5, 45.0, 280.0, 85.0, 210.0, 360.0, 480.0, 40.0, 310.0],
    'cout_total_5_usages': [2500, 1500, 400, 2000, 800, 1200, 1900, 2800, 500, 1100],
    'qualite_isolation_murs': ['insuffisante', 'bonne', 'très bonne', 'moyenne', 'très bonne', 'moyenne', 'insuffisante', 'insuffisante', 'très bonne', 'moyenne'],
    'qualite_isolation_menuiseries': ['moyenne', 'bonne', 'très bonne', 'moyenne', 'bonne', 'moyenne', 'insuffisante', 'moyenne', 'très bonne', 'insuffisante'],
    'type_ventilation': ['Naturelle', 'VMC simple', 'VMC double', 'VMC simple', 'VMC double', 'Naturelle', 'Naturelle', 'VMC simple', 'VMC double', 'Naturelle'],
    'type_energie_principale_chauffage': ['Fioul', 'Gaz', 'Électricité', 'Gaz', 'Électricité', 'Gaz', 'Fioul', 'Électricité', 'Électricité', 'Gaz'],
    'besoin_chauffage': [15000, 8000, 2000, 10000, 3500, 7000, 12000, 14000, 2500, 6000],
    'besoin_ecs': [2500, 3000, 1500, 2800, 2000, 2200, 2600, 2400, 1800, 1600],
    'type_batiment': ['appartement', 'maison', 'appartement', 'appartement', 'appartement', 'appartement', 'maison', 'appartement', 'appartement', 'appartement'],
    'nombre_appartement': [12, 1, 8, 20, 50, 15, 1, 10, 40, 6]
}

df_source = pd.DataFrame(data_test)
# Sauvegarde en CSV pour simuler le fichier réel
df_source.to_csv('big_dpe_simulated.csv', index=False)
print("✅ Fichier 'big_dpe_simulated.csv' créé pour le test !")
df_source

In [ ]:
# Simulation du nettoyage
df_test = df_source.copy()
# 1. On enlève les doublons sur numero_dpe
df_test = df_test.drop_duplicates(subset=['numero_dpe'])
df_test


In [ ]:
# 2. On enlève les DPE qui n'ont pas d'étiquette (important pour les calculs)
df_test = df_test.dropna(subset=['etiquette_dpe'])
df_test['etiquette_dpe']

In [ ]:
# 3. Conversion en catégories (pour économiser la RAM)
df_test['etiquette_dpe'] = df_test['etiquette_dpe'].astype('category')
df_test

In [ ]:
shape=df_test.shape
shape

In [ ]:

df_test.head(3)

In [ ]:
# Création de la colonne par défaut
df_test['statut_juridique'] = 'Privé (Individuel)'
df_test['statut_juridique']

In [ ]:
# Si numero_rpls_logement n'est pas vide -> C'est du Social
condition_social = df_test['numero_rpls_logement'].notna()
condition_social

In [ ]:
df_test.loc[condition_social, 'statut_juridique'] = 'Social'
df_test['statut_juridique']

In [ ]:
# Si numero_immatriculation_copropriete n'est pas vide -> C'est du Privé (Copro)
condition_copro = df_test['numero_immatriculation_copropriete'].notna()
condition_copro

In [ ]:
df_test.loc[condition_copro, 'statut_juridique'] = 'Privé (Copro)'
df_test['statut_juridique']

In [ ]:

# On regarde le résultat
df_test[['numero_dpe', 'numero_rpls_logement', 'statut_juridique']].head(5)

In [ ]:
# On considère rénové si l'étiquette est A, B ou C
df_test['is_renovated'] = df_test['etiquette_dpe'].isin(['A', 'B', 'C','D']).astype(int)

df_test['is_renovated']

In [ ]:
# # On calcule la moyenne par Code Postal et par Statut
# renovation_by_postal_and_status = df_test.groupby(['code_postal_brut', 'statut_juridique'])['is_renovated'].mean()
# renovation_by_postal_and_status

###Cellule 5 : La méthode "Senior" (Compacte avec Multi-index)


In [ ]:
df_test["total"]=df_test['numero_dpe'].notna().astype(int)


# 1. On calcule la moyenne par CP et Statut
renovation_by_postal_and_status = df_test.groupby(
    ['code_postal_brut', 'statut_juridique']
)['is_renovated'].agg(['mean'])

renovation_by_postal_and_status
renovation_by_postal_and_status2 = df_test.groupby(
    ['code_postal_brut', 'statut_juridique']
)['total'].agg(['sum'])
renovation_by_postal_and_status2

In [ ]:
# 2. On "dé-pile" (unstack) pour avoir les statuts en colonnes
# fill_value=0 est crucial si un quartier n'a pas de logements "Sociaux" par exemple
renovation_pivot = renovation_by_postal_and_status.unstack(fill_value=0)
renovation_pivot

In [ ]:
renovation_pivot[('mean', 'Social')]

In [ ]:
df_test['code_postal_brut'].map(
        renovation_pivot[('mean', 'Social')]
    )

In [ ]:
# 4. Le Mapping complexe
# Ici on va chercher précisément la colonne ('mean', 'Social')
if ('mean', 'Social') in renovation_pivot.columns:
    df_test['pourcentage_renover_social_expert'] = df_test['code_postal_brut'].map(
        renovation_pivot[('mean', 'Social')]
    )

print("\nRésultat de la méthode EXPERT (ajout de la colonne via Map direct) :")
df_test[['code_postal_brut', 'statut_juridique', 'pourcentage_renover_social_expert']].head()

In [ ]:
# On compare les deux colonnes créées
comparaison = df_test[df_test['statut_juridique'] == 'Social'][
    ['code_postal_brut', 'pourcentage_renover_social_expert']
]

print("Comparaison des deux méthodes (pour les logements sociaux) :")
display(comparaison.head())

In [ ]:




# On transforme le résultat du groupby en tableau large
renovation_pivot = renovation_by_postal_and_status.unstack(fill_value=0)*100
renovation_pivot
df_test

In [ ]:
# 1. On extrait juste la série des taux Sociaux
serie_taux_social = renovation_pivot[('mean', 'Social')]
serie_taux_social

In [ ]:
renovation_pivot

In [ ]:

df_test['pourcentage_renover_social'] = df_test['code_postal_brut'].map(
        renovation_pivot.get_level_values(1).map({
            'Social': renovation_pivot.loc[:, ('mean', 'Social')] * 100
        })
        if ('mean', 'Social') in renovation_pivot.columns else 0
    )


In [ ]:
# 1. On initialise nos dictionnaires (Vides au départ)
social_renovation_dict = {}
private_renovation_dict = {}

In [ ]:
# 2. On récupère la liste de tous les codes postaux uniques
codes_postaux = df_test['code_postal_brut'].unique()
codes_postaux

In [ ]:
df_test['statut_juridique']

In [ ]:
(df_test['code_postal_brut'] == 75008)

In [ ]:
(df_test['statut_juridique'] == 'Social')

In [ ]:
(df_test['code_postal_brut'].head(1) == 75008) & (df_test['statut_juridique'].head(1) == 'Social')
(df_test['statut_juridique'].head(1) == 'Social')
(df_test['code_postal_brut'].head(1) == 75008)

In [ ]:
False or False

In [ ]:
df_test[(df_test['code_postal_brut'] == 75008) & (df_test['statut_juridique'] == 'Social')]

In [ ]:
social_houses = df_test[(df_test['code_postal_brut'] == 75008) & (df_test['statut_juridique'] == 'Social')]
social_houses

In [ ]:


for postal in codes_postaux:
    # --- PARTIE SOCIALE ---
    # On filtre : ce code postal ET statut "Social"
    social_houses = df_test[(df_test['code_postal_brut'] == postal) & (df_test['statut_juridique'] == 'Social')]
    
    if len(social_houses) > 0:
        # Calcul : (Somme des rénovés / Nombre total) * 100
        social_rate = (social_houses['is_renovated'].sum() / len(social_houses)) * 100
        social_renovation_dict[postal] = round(social_rate, 1)
    else:
        social_renovation_dict[postal] = 0.0

    # --- PARTIE PRIVÉE ---
    # On filtre : ce code postal ET statut n'est pas "Social"
    private_houses = df_test[(df_test['code_postal_brut'] == postal) & (df_test['statut_juridique'] != 'Social')]
    
    if len(private_houses) > 0:
        private_rate = (private_houses['is_renovated'].sum() / len(private_houses)) * 100
        private_renovation_dict[postal] = round(private_rate, 1)
    else:
        private_renovation_dict[postal] = 0.0

print("Dictionnaire Social (CP: % rénové) :", social_renovation_dict)
print("Dictionnaire Privé (CP: % rénové) :", private_renovation_dict)

In [ ]:
df_test[['numero_dpe', 'etiquette_dpe', 'is_renovated']].head()

In [ ]:
import os
print('os.path("./data_light_output/dpe03existant_paris_light.csv"): ', os.path("./data_light_output/dpe03existant_paris_light.csv"))